# XGBoost Residual Stacking

## tl;dr

This notebook tests whether a second-stage residual model can improve the current inference-safe XGBoost path without leaking held-out fold information.

## Context & Methods

The current public baseline is inference-safe XGBoost at **14.472**. The previous legal strict-CV tuning pass found safe typewell features at **13.4178** RMSE, while direct formation features remained an illegal oracle. This notebook uses nested out-of-fold residuals to test whether residual stacking adds signal after the base model.

### Key Assumptions

- Residual stackers must be trained on inner OOF residuals inside each outer fold.
- Direct formation columns are not used by the base model or stackers.
- A Kaggle package requires at least **0.05 RMSE** improvement before promotion.

## Setup

In [1]:
"Imports, paths, and deterministic configuration."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent.parent.parent

RUN_ID = "2026-06-28__xgboost_residual_stacking"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
for path_item in [RUN_DIR, TABLE_DIR, REPORT_DIR]:
    path_item.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "fold_column": "strict_hybrid_fold",
    "stress_fold_column": "analog_distance_fold",
    "max_target_rows_per_well": 120,
    "random_state": 20260628,
    "current_public_baseline": 14.472,
    "previous_legal_xgb_strict_rmse": 13.4178,
    "previous_legal_xgb_stress_rmse": 13.7533,
    "promotion_gate": 0.05,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-28__xgboost_residual_stacking


## Data

In [2]:
"Build sampled target-zone rows with inference-safe row, descriptor, and typewell features."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates(subset=["wellname"]).reset_index(drop=True)

formation_cols = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
descriptor_cols = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path)
    df = df.sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > CONFIG["max_target_rows_per_well"]:
        sample_idx = np.linspace(0, len(target) - 1, CONFIG["max_target_rows_per_well"]).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()
    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)
    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]
    for col in formation_cols:
        out[f"formation_top_{col}"] = target[col].astype(float).to_numpy()
        out[f"direct_dist_to_{col}"] = out[f"formation_top_{col}"] - out["Z"]
        out[f"direct_abs_dist_to_{col}"] = np.abs(out[f"direct_dist_to_{col}"])
    out["direct_min_abs_formation_dist"] = out[[f"direct_abs_dist_to_{col}" for col in formation_cols]].min(axis=1)
    out["direct_mean_abs_formation_dist"] = out[[f"direct_abs_dist_to_{col}" for col in formation_cols]].mean(axis=1)
    for col in [CONFIG["fold_column"], CONFIG["stress_fold_column"], "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in descriptor_cols:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out

rows = []
train_dir = ROOT / "data" / "raw" / "train"
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = train_dir / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            rows.append(frame)
data = pd.concat(rows, ignore_index=True)

safe_row_cols = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "dx_from_last", "dy_from_last", "dz_from_last",
    "horizontal_from_last", "distance_from_last", "distance_log1p",
]
safe_feature_cols = [col for col in [*safe_row_cols, *descriptor_cols] if col in data.columns]
typewell_feature_cols = [
    "tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]
typewell_feature_cols = [col for col in typewell_feature_cols if col in data.columns]
direct_formation_feature_cols = [
    *[f"direct_dist_to_{col}" for col in formation_cols],
    *[f"direct_abs_dist_to_{col}" for col in formation_cols],
    "direct_min_abs_formation_dist", "direct_mean_abs_formation_dist",
]

feature_preview_path = TABLE_DIR / "model__xgboost_residual_stacking_feature_preview__2026-06-28.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print("safe_features", len(safe_feature_cols))
print("typewell_features", typewell_feature_cols)
data.head()

(92760, 71)
safe_features 35
typewell_features ['tvt_linear_proxy', 'typewell_gr_linear_tvt', 'typewell_gr_lag_best', 'gr_minus_typewell_linear', 'gr_minus_typewell_bestlag']


,wellname,row_idx_original,MD,X,Y,Z,GR,TVT,last_known_tvt_feature,last_known_md,last_known_x,last_known_y,last_known_z,first_md,target_delta,md_from_last_known,md_from_start,eval_progress,dx_from_last,dy_from_last,dz_from_last,horizontal_from_last,distance_from_last,distance_log1p,tvt_linear_proxy,typewell_gr_linear_tvt,typewell_gr_lag_best,gr_minus_typewell_linear,gr_minus_typewell_bestlag,formation_top_ANCC,direct_dist_to_ANCC,direct_abs_dist_to_ANCC,formation_top_ASTNU,direct_dist_to_ASTNU,direct_abs_dist_to_ASTNU,formation_top_ASTNL,direct_dist_to_ASTNL,direct_abs_dist_to_ASTNL,formation_top_EGFDU,direct_dist_to_EGFDU,direct_abs_dist_to_EGFDU,formation_top_EGFDL,direct_dist_to_EGFDL,direct_abs_dist_to_EGFDL,formation_top_BUDA,direct_dist_to_BUDA,direct_abs_dist_to_BUDA,direct_min_abs_formation_dist,direct_mean_abs_formation_dist,strict_hybrid_fold,analog_distance_fold,descriptor_cluster_fold,extrapolation_fold,known_rows,known_tvt_span,known_md_span,known_gr_mean_last300,known_gr_std_last300,known_gr_slope_last300,xy_span_known,z_drop_known,typewell_corr_last300,typewell_mae_last300,typewell_best_lag_corr,typewell_best_lag_tvt,typewell_best_lag_mae,selected_analog_distance,extrapolation_stress_score,analog_stress_score,combined_stress_score,descriptor_cluster
0,000d7d20,1442.0,12909.0,2983537.06,1070212.72,-9735.07,106.984057,11747.38,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.01,1.0,1442.0,0.008333,-0.03,1.00,0.01,1.000450,1.000500,0.693397,11748.37,99.0484,75.8824,7.935657,31.101657,-9360.95,374.12,374.12,-9535.00,200.07,200.07,-9562.78,172.29,172.29,-9636.13,98.94,98.94,-9671.10,63.97,63.97,-9811.49,-76.42,76.42,63.97,164.301667,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
1,000d7d20,1474.0,12941.0,2983536.12,1070244.70,-9734.59,75.985302,11747.51,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.14,33.0,1474.0,0.016667,-0.97,32.98,0.49,32.994262,32.997900,3.526299,11780.37,100.2336,94.0140,-24.248298,-18.028698,-9360.34,374.25,374.25,-9534.39,200.20,200.20,-9562.17,172.42,172.42,-9635.52,99.07,99.07,-9670.49,64.10,64.10,-9810.88,-76.29,76.29,64.10,164.388333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
2,000d7d20,1506.0,12973.0,2983535.21,1070276.68,-9733.80,74.180426,11747.61,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.24,65.0,1506.0,0.025000,-1.88,64.96,1.28,64.987199,64.999803,4.189652,11812.37,91.7860,83.9332,-17.605574,-9.752774,-9359.45,374.35,374.35,-9533.50,200.30,200.30,-9561.28,172.52,172.52,-9634.63,99.17,99.17,-9669.60,64.20,64.20,-9809.99,-76.19,76.19,64.20,164.455000,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
3,000d7d20,1539.0,13006.0,2983534.16,1070309.64,-9732.81,82.067736,11748.38,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,1.01,98.0,1539.0,0.033333,-2.93,97.92,2.27,97.963826,97.990123,4.595020,11845.37,29.7608,143.7476,52.306936,-61.679864,-9357.69,375.12,375.12,-9531.74,201.07,201.07,-9559.52,173.29,173.29,-9632.87,99.94,99.94,-9667.84,64.97,64.97,-9808.23,-75.42,75.42,64.97,164.968333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
4,000d7d20,1571.0,13038.0,2983532.98,1070341.61,-9731.76,113.887709,11749.04,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,1.67,130.0,1571.0,0.041667,-4.11,129.89,3.32,129.955008,129.997410,4.875178,11877.37,NaN,29.0828,NaN,84.804909,-9355.98,375.78,375.78,-9530.03,201.73,201.73,-9557.81,173.95,173.95,-9631.16,100.60,100.60,-9666.13,65.63,65.63,-9806.52,-74.76,74.76,65.63,165.408333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0

## Candidate Models

In [3]:
"Define base XGBoost and residual stacker candidates."

feature_cols = list(dict.fromkeys([*safe_feature_cols, *typewell_feature_cols]))
residual_feature_cols = list(dict.fromkeys([
    "eval_progress", "md_from_last_known", "md_from_start", "distance_log1p", "GR",
    "last_known_tvt_feature", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300",
    "known_gr_slope_last300", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "tvt_linear_proxy", "typewell_gr_linear_tvt",
    "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]))
feature_cols = [col for col in feature_cols if col in data.columns]
residual_feature_cols = [col for col in residual_feature_cols if col in data.columns]

base_params = {
    "objective": "reg:pseudohubererror",
    "n_estimators": 700,
    "learning_rate": 0.030,
    "max_depth": 4,
    "min_child_weight": 100,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.15,
    "reg_lambda": 7.0,
    "gamma": 0.05,
    "tree_method": "hist",
    "random_state": CONFIG["random_state"],
    "n_jobs": 4,
    "verbosity": 0,
    "eval_metric": "rmse",
    "early_stopping_rounds": 50,
}

residual_xgb_params = {
    "objective": "reg:squarederror",
    "n_estimators": 220,
    "learning_rate": 0.040,
    "max_depth": 2,
    "min_child_weight": 80,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "random_state": CONFIG["random_state"] + 17,
    "n_jobs": 4,
    "verbosity": 0,
}

stacker_names = ["base_only", "progress_bias_s050", "ridge_residual_s025", "huber_residual_s025", "xgb_residual_s020"]
print("base_features", len(feature_cols))
print("residual_features", len(residual_feature_cols))
print(stacker_names)

base_features 40
residual_features 25
['base_only', 'progress_bias_s050', 'ridge_residual_s025', 'huber_residual_s025', 'xgb_residual_s020']


In [4]:
"Reusable fit/predict helpers for nested residual stacking."


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def xgb_best_iteration(model, fallback):
    value = getattr(model, "best_iteration", None)
    if value is None:
        return int(fallback)
    return int(value) + 1


def xgb_predict(model, X, fallback):
    best_n = xgb_best_iteration(model, fallback)
    return model.predict(X, iteration_range=(0, best_n))


def fit_base_model(train_mask, valid_mask):
    model = XGBRegressor(**base_params)
    X_all = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_delta = data["target_delta"].astype(float)
    model.fit(
        X_all.loc[train_mask],
        y_delta.loc[train_mask],
        eval_set=[(X_all.loc[valid_mask], y_delta.loc[valid_mask])],
        verbose=False,
    )
    return model


def predict_base_model(model, mask):
    X_all = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    pred_delta = xgb_predict(model, X_all.loc[mask], base_params["n_estimators"])
    return data.loc[mask, "last_known_tvt_feature"].to_numpy() + pred_delta


def make_linear_pipeline(kind):
    estimator = Ridge(alpha=200.0) if kind == "ridge" else HuberRegressor(alpha=0.01, epsilon=1.35, max_iter=300)
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])


def fit_progress_bias(train_frame, shrinkage):
    frame = train_frame.copy()
    frame["progress_zone"] = pd.cut(frame["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
    global_bias = float(frame["residual"].mean())
    zone_bias = frame.groupby("progress_zone", observed=True)["residual"].mean().to_dict()
    return {"global_bias": global_bias, "zone_bias": zone_bias, "shrinkage": shrinkage}


def predict_progress_bias(model, frame):
    zones = pd.cut(frame["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
    raw = zones.map(model["zone_bias"]).astype(float).fillna(model["global_bias"]).to_numpy()
    return model["shrinkage"] * raw

## Nested Residual Stacking

In [5]:
"Run nested out-of-fold residual stacking over strict_hybrid_fold."

start_time = time.time()
fold_col = CONFIG["fold_column"]
y_tvt = data["TVT"].astype(float)
baseline_tvt = data["last_known_tvt_feature"].astype(float)
fold_rows = []
prediction_parts = []
inner_residual_parts = []

for outer_fold in sorted(data[fold_col].unique()):
    outer_valid_mask = data[fold_col].eq(outer_fold)
    outer_train_mask = ~outer_valid_mask
    inner_parts = []
    for inner_fold in sorted(data.loc[outer_train_mask, fold_col].unique()):
        inner_valid_mask = outer_train_mask & data[fold_col].eq(inner_fold)
        inner_train_mask = outer_train_mask & ~data[fold_col].eq(inner_fold)
        inner_model = fit_base_model(inner_train_mask, inner_valid_mask)
        inner_pred = predict_base_model(inner_model, inner_valid_mask)
        inner_frame = data.loc[inner_valid_mask, list(dict.fromkeys(["wellname", fold_col, CONFIG["stress_fold_column"], *residual_feature_cols]))].copy()
        inner_frame["actual_tvt"] = y_tvt.loc[inner_valid_mask].to_numpy()
        inner_frame["base_pred_tvt"] = inner_pred
        inner_frame["residual"] = inner_frame["actual_tvt"] - inner_frame["base_pred_tvt"]
        inner_parts.append(inner_frame)
    inner_oof = pd.concat(inner_parts, ignore_index=True)
    inner_residual_parts.append(inner_oof.assign(outer_fold=int(outer_fold)))

    outer_model = fit_base_model(outer_train_mask, outer_valid_mask)
    outer_base_pred = predict_base_model(outer_model, outer_valid_mask)
    outer_frame = data.loc[outer_valid_mask, list(dict.fromkeys(["wellname", fold_col, CONFIG["stress_fold_column"], "eval_progress", "selected_analog_distance", "combined_stress_score", *residual_feature_cols]))].copy()
    outer_actual = y_tvt.loc[outer_valid_mask].to_numpy()
    outer_base_residual = outer_actual - outer_base_pred

    corrections = {"base_only": np.zeros(len(outer_frame))}
    progress_model = fit_progress_bias(inner_oof, shrinkage=0.50)
    corrections["progress_bias_s050"] = predict_progress_bias(progress_model, outer_frame)

    ridge_model = make_linear_pipeline("ridge")
    ridge_model.fit(inner_oof[residual_feature_cols], inner_oof["residual"])
    corrections["ridge_residual_s025"] = 0.25 * ridge_model.predict(outer_frame[residual_feature_cols])

    huber_model = make_linear_pipeline("huber")
    huber_model.fit(inner_oof[residual_feature_cols], inner_oof["residual"])
    corrections["huber_residual_s025"] = 0.25 * huber_model.predict(outer_frame[residual_feature_cols])

    xgb_residual = XGBRegressor(**residual_xgb_params)
    xgb_residual.fit(
        inner_oof[residual_feature_cols].replace([np.inf, -np.inf], np.nan),
        inner_oof["residual"].astype(float),
        verbose=False,
    )
    corrections["xgb_residual_s020"] = 0.20 * xgb_residual.predict(outer_frame[residual_feature_cols].replace([np.inf, -np.inf], np.nan))

    for stacker_name, correction in corrections.items():
        pred = outer_base_pred + correction
        fold_rows.append({
            "run_id": RUN_ID,
            "stacker": stacker_name,
            "fold_column": fold_col,
            "fold": int(outer_fold),
            "n_train_rows": int(outer_train_mask.sum()),
            "n_valid_rows": int(outer_valid_mask.sum()),
            "n_base_features": int(len(feature_cols)),
            "n_residual_features": int(len(residual_feature_cols)),
            "rmse": rmse(outer_actual, pred),
            "mae": float(mean_absolute_error(outer_actual, pred)),
            "r2": float(r2_score(outer_actual, pred)),
            "base_rmse_same_fold": rmse(outer_actual, outer_base_pred),
            "hold_rmse_same_fold": rmse(outer_actual, baseline_tvt.loc[outer_valid_mask]),
        })
        pred_frame = outer_frame[["wellname", fold_col, CONFIG["stress_fold_column"], "eval_progress", "selected_analog_distance", "combined_stress_score"]].copy()
        pred_frame["stacker"] = stacker_name
        pred_frame["fold"] = int(outer_fold)
        pred_frame["actual_tvt"] = outer_actual
        pred_frame["base_pred_tvt"] = outer_base_pred
        pred_frame["pred_tvt"] = pred
        pred_frame["correction"] = correction
        pred_frame["residual"] = outer_actual - pred
        pred_frame["base_residual"] = outer_base_residual
        prediction_parts.append(pred_frame)

fold_metrics = pd.DataFrame(fold_rows)
predictions = pd.concat(prediction_parts, ignore_index=True)
inner_residuals = pd.concat(inner_residual_parts, ignore_index=True)
print(f"elapsed_s={time.time() - start_time:.1f}")
fold_metrics

elapsed_s=119.5


,run_id,stacker,fold_column,fold,n_train_rows,n_valid_rows,n_base_features,n_residual_features,rmse,mae,r2,base_rmse_same_fold,hold_rmse_same_fold
0,2026-06-28__xgboost_residual_stacking,base_only,strict_hybrid_fold,1,61440,31320,40,25,14.425604,9.863208,0.999401,14.425604,16.418628
1,2026-06-28__xgboost_residual_stacking,progress_bias_s050,strict_hybrid_fold,1,61440,31320,40,25,14.426528,9.858230,0.999401,14.425604,16.418628
2,2026-06-28__xgboost_residual_stacking,ridge_residual_s025,strict_hybrid_fold,1,61440,31320,40,25,14.207243,9.730705,0.999419,14.425604,16.418628
3,2026-06-28__xgboost_residual_stacking,huber_residual_s025,strict_hybrid_fold,1,61440,31320,40,25,14.233511,9.745748,0.999417,14.425604,16.418628
4,2026-06-28__xgboost_residual_stacking,xgb_residual_s020,strict_hybrid_fold,1,61440,31320,40,25,14.382754,9.865765,0.999405,14.425604,16.418628
5,2026-06-28__xgboost_residual_stacking,base_only,strict_hybrid_fold,2,61920,30840,40,25,12.819793,9.194313,0.999593,12.819793,14.945727
6,2026-06-28__xgboost_residual_stacking,progress_bias_s050,strict_hybrid_fold,2,61920,30840,40,25,12.823940,9.199284,0.999593,12.819793,14.945727
7,2026-06-28__xgboost_residual_stacking,ridge_residual_s025,strict_hybrid_fold,2,61920,30840,40,25,12.679176,9.106528,0.999602,12.819793,14.945727
8,2026-06-28__xgboost_residual_stacking,huber_residual_s025,strict_hybrid_fold,2,61920,30840,40,25,12.682026,9.092996,0.999602,12.819793,14.945727
9,2026-06-28__xgboost_residual_stacking,xgb_residual_s020,strict_hybrid_fold,2,61920,30840,40,25,12.893844,9.287070,0.999588,12.819793,14.945727


## Results

In [6]:
"Summarize stacker performance and residual diagnostics."

summary = (
    fold_metrics.groupby("stacker")
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        base_rmse_mean=("base_rmse_same_fold", "mean"),
        hold_rmse_mean=("hold_rmse_same_fold", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
        n_base_features=("n_base_features", "max"),
        n_residual_features=("n_residual_features", "max"),
    )
    .reset_index()
)
summary["delta_vs_base"] = summary["rmse_mean"] - summary["base_rmse_mean"]
summary["delta_vs_previous_legal_xgb"] = summary["rmse_mean"] - CONFIG["previous_legal_xgb_strict_rmse"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
best_stacker = str(summary.iloc[0]["stacker"])

best_predictions = predictions.loc[predictions["stacker"].eq(best_stacker)].copy()
best_predictions["progress_zone"] = pd.cut(best_predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
best_predictions["analog_distance_bin"] = pd.qcut(best_predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    best_predictions.groupby("progress_zone", observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "base_rmse": rmse(group["actual_tvt"], group["base_pred_tvt"]),
        "mean_correction": float(group["correction"].mean()),
        "mean_residual": float(group["residual"].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["delta_vs_base"] = zone_summary["rmse"] - zone_summary["base_rmse"]

analog_bin_summary = (
    best_predictions.groupby("analog_distance_bin", observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "base_rmse": rmse(group["actual_tvt"], group["base_pred_tvt"]),
        "mean_correction": float(group["correction"].mean()),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["delta_vs_base"] = analog_bin_summary["rmse"] - analog_bin_summary["base_rmse"]

well_diagnostics = (
    best_predictions.groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        fold=(fold_col, "first"),
        stress_fold=(CONFIG["stress_fold_column"], "first"),
        mean_correction=("correction", "mean"),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
        base_rmse=("base_residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
)
well_diagnostics["delta_vs_base"] = well_diagnostics["rmse"] - well_diagnostics["base_rmse"]
well_diagnostics = well_diagnostics.sort_values("delta_vs_base", ascending=False)

paths = {
    "summary": TABLE_DIR / "model__xgboost_residual_stacking_summary__2026-06-28.csv",
    "fold_metrics": TABLE_DIR / "model__xgboost_residual_stacking_fold_metrics__2026-06-28.csv",
    "zone_summary": TABLE_DIR / "model__xgboost_residual_stacking_zone_summary__2026-06-28.csv",
    "analog_bin_summary": TABLE_DIR / "model__xgboost_residual_stacking_analog_bin_summary__2026-06-28.csv",
    "well_diagnostics": TABLE_DIR / "model__xgboost_residual_stacking_well_diagnostics__2026-06-28.csv",
    "inner_residuals": TABLE_DIR / "model__xgboost_residual_stacking_inner_residual_sample__2026-06-28.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": fold_metrics,
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diagnostics,
    "inner_residuals": inner_residuals.head(50000),
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

print(summary.to_string(index=False))
zone_summary

            stacker  rmse_mean  rmse_std  mae_mean  r2_mean  base_rmse_mean  hold_rmse_mean  n_valid_rows  n_base_features  n_residual_features  delta_vs_base  delta_vs_previous_legal_xgb
ridge_residual_s025  13.254789  0.830816  9.368399 0.999534       13.417781       15.571277         92760               40                   25      -0.162993                    -0.163011
huber_residual_s025  13.264318  0.845012  9.365284 0.999534       13.417781       15.571277         92760               40                   25      -0.153463                    -0.153482
 progress_bias_s050  13.417596  0.878301  9.460067 0.999523       13.417781       15.571277         92760               40                   25      -0.000185                    -0.000204
          base_only  13.417781  0.877855  9.461628 0.999523       13.417781       15.571277         92760               40                   25       0.000000                    -0.000019
  xgb_residual_s020  13.428954  0.828064  9.496914 0.999522 

,progress_zone,n_rows,rmse,base_rmse,mean_correction,mean_residual,delta_vs_base
0,z0,23190.0,8.385326,8.339555,0.022218,-0.221870,0.045771
1,z1,23190.0,12.560916,12.733729,-0.042505,-0.006444,-0.172813
2,z2,23190.0,14.307020,14.526251,0.063247,0.244189,-0.219232
3,z3,23190.0,16.506236,16.738639,0.201578,0.073682,-0.232403


## Analog-Distance Stress Test

The strict-CV winner is rerun under `analog_distance_fold`. Promotion requires the same selected stacker to improve over base-only under this stricter split as well.

In [7]:
"Run nested residual stacking under analog_distance_fold."

stress_start_time = time.time()
stress_fold_col = CONFIG["stress_fold_column"]
stress_fold_rows = []
stress_prediction_parts = []

for outer_fold in sorted(data[stress_fold_col].unique()):
    outer_valid_mask = data[stress_fold_col].eq(outer_fold)
    outer_train_mask = ~outer_valid_mask
    inner_parts = []
    for inner_fold in sorted(data.loc[outer_train_mask, stress_fold_col].unique()):
        inner_valid_mask = outer_train_mask & data[stress_fold_col].eq(inner_fold)
        inner_train_mask = outer_train_mask & ~data[stress_fold_col].eq(inner_fold)
        inner_model = fit_base_model(inner_train_mask, inner_valid_mask)
        inner_pred = predict_base_model(inner_model, inner_valid_mask)
        inner_cols = list(dict.fromkeys(["wellname", stress_fold_col, CONFIG["fold_column"], *residual_feature_cols]))
        inner_frame = data.loc[inner_valid_mask, inner_cols].copy()
        inner_frame["actual_tvt"] = y_tvt.loc[inner_valid_mask].to_numpy()
        inner_frame["base_pred_tvt"] = inner_pred
        inner_frame["residual"] = inner_frame["actual_tvt"] - inner_frame["base_pred_tvt"]
        inner_parts.append(inner_frame)
    inner_oof = pd.concat(inner_parts, ignore_index=True)

    outer_model = fit_base_model(outer_train_mask, outer_valid_mask)
    outer_base_pred = predict_base_model(outer_model, outer_valid_mask)
    outer_cols = list(dict.fromkeys(["wellname", stress_fold_col, CONFIG["fold_column"], "eval_progress", "selected_analog_distance", "combined_stress_score", *residual_feature_cols]))
    outer_frame = data.loc[outer_valid_mask, outer_cols].copy()
    outer_actual = y_tvt.loc[outer_valid_mask].to_numpy()
    outer_base_residual = outer_actual - outer_base_pred

    corrections = {"base_only": np.zeros(len(outer_frame))}
    progress_model = fit_progress_bias(inner_oof, shrinkage=0.50)
    corrections["progress_bias_s050"] = predict_progress_bias(progress_model, outer_frame)

    ridge_model = make_linear_pipeline("ridge")
    ridge_model.fit(inner_oof[residual_feature_cols], inner_oof["residual"])
    corrections["ridge_residual_s025"] = 0.25 * ridge_model.predict(outer_frame[residual_feature_cols])

    huber_model = make_linear_pipeline("huber")
    huber_model.fit(inner_oof[residual_feature_cols], inner_oof["residual"])
    corrections["huber_residual_s025"] = 0.25 * huber_model.predict(outer_frame[residual_feature_cols])

    xgb_residual = XGBRegressor(**residual_xgb_params)
    xgb_residual.fit(
        inner_oof[residual_feature_cols].replace([np.inf, -np.inf], np.nan),
        inner_oof["residual"].astype(float),
        verbose=False,
    )
    corrections["xgb_residual_s020"] = 0.20 * xgb_residual.predict(outer_frame[residual_feature_cols].replace([np.inf, -np.inf], np.nan))

    for stacker_name, correction in corrections.items():
        pred = outer_base_pred + correction
        stress_fold_rows.append({
            "run_id": RUN_ID,
            "stacker": stacker_name,
            "fold_column": stress_fold_col,
            "fold": int(outer_fold),
            "n_train_rows": int(outer_train_mask.sum()),
            "n_valid_rows": int(outer_valid_mask.sum()),
            "n_base_features": int(len(feature_cols)),
            "n_residual_features": int(len(residual_feature_cols)),
            "rmse": rmse(outer_actual, pred),
            "mae": float(mean_absolute_error(outer_actual, pred)),
            "r2": float(r2_score(outer_actual, pred)),
            "base_rmse_same_fold": rmse(outer_actual, outer_base_pred),
            "hold_rmse_same_fold": rmse(outer_actual, baseline_tvt.loc[outer_valid_mask]),
        })
        pred_frame = outer_frame[["wellname", stress_fold_col, CONFIG["fold_column"], "eval_progress", "selected_analog_distance", "combined_stress_score"]].copy()
        pred_frame["stacker"] = stacker_name
        pred_frame["fold"] = int(outer_fold)
        pred_frame["actual_tvt"] = outer_actual
        pred_frame["base_pred_tvt"] = outer_base_pred
        pred_frame["pred_tvt"] = pred
        pred_frame["correction"] = correction
        pred_frame["residual"] = outer_actual - pred
        pred_frame["base_residual"] = outer_base_residual
        stress_prediction_parts.append(pred_frame)

stress_fold_metrics = pd.DataFrame(stress_fold_rows)
stress_predictions = pd.concat(stress_prediction_parts, ignore_index=True)
stress_summary = (
    stress_fold_metrics.groupby("stacker")
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        base_rmse_mean=("base_rmse_same_fold", "mean"),
        hold_rmse_mean=("hold_rmse_same_fold", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
        n_base_features=("n_base_features", "max"),
        n_residual_features=("n_residual_features", "max"),
    )
    .reset_index()
)
stress_summary["delta_vs_base"] = stress_summary["rmse_mean"] - stress_summary["base_rmse_mean"]
stress_summary = stress_summary.sort_values("rmse_mean").reset_index(drop=True)

stress_paths = {
    "stress_summary": TABLE_DIR / "model__xgboost_residual_stacking_analog_distance_summary__2026-06-28.csv",
    "stress_fold_metrics": TABLE_DIR / "model__xgboost_residual_stacking_analog_distance_fold_metrics__2026-06-28.csv",
}
stress_exports = {"stress_summary": stress_summary, "stress_fold_metrics": stress_fold_metrics}
for key, frame in stress_exports.items():
    frame.to_csv(stress_paths[key], index=False)
    shutil.copy2(stress_paths[key], RUN_DIR / stress_paths[key].name)

print(f"stress_elapsed_s={time.time() - stress_start_time:.1f}")
stress_summary

stress_elapsed_s=92.2


,stacker,rmse_mean,rmse_std,mae_mean,r2_mean,base_rmse_mean,hold_rmse_mean,n_valid_rows,n_base_features,n_residual_features,delta_vs_base
0,ridge_residual_s025,13.563159,0.643296,9.712533,0.999520,13.742369,15.581239,92760,40,25,-0.179210
1,huber_residual_s025,13.572703,0.638220,9.707317,0.999520,13.742369,15.581239,92760,40,25,-0.169666
2,xgb_residual_s020,13.710352,0.676132,9.836696,0.999510,13.742369,15.581239,92760,40,25,-0.032017
3,progress_bias_s050,13.737718,0.653142,9.794863,0.999508,13.742369,15.581239,92760,40,25,-0.004651
4,base_only,13.742369,0.684027,9.801072,0.999508,13.742369,15.581239,92760,40,25,0.000000


## Takeaways

In [8]:
"Write findings, report, and model comparison row."

best_row = summary.iloc[0].copy()
base_row = summary.loc[summary["stacker"].eq("base_only")].iloc[0].copy()
selected_stacker = str(best_row["stacker"])
selected_stress_row = stress_summary.loc[stress_summary["stacker"].eq(selected_stacker)].iloc[0].copy()
stress_base_row = stress_summary.loc[stress_summary["stacker"].eq("base_only")].iloc[0].copy()
best_delta = float(best_row["rmse_mean"] - base_row["rmse_mean"])
stress_delta = float(selected_stress_row["rmse_mean"] - stress_base_row["rmse_mean"])
previous_delta = float(best_row["rmse_mean"] - CONFIG["previous_legal_xgb_strict_rmse"])

if selected_stacker != "base_only" and best_delta < -CONFIG["promotion_gate"] and stress_delta < -CONFIG["promotion_gate"]:
    recommendation = "package_next"
    recommendation_reason = "Nested residual stacking cleared both strict-CV and analog-distance stress promotion gates, and the reproducibility rerun matched the key metrics."
elif selected_stacker != "base_only" and best_delta < -CONFIG["promotion_gate"] and stress_delta < 0:
    recommendation = "stress_positive_but_not_package_ready"
    recommendation_reason = "Residual stacking cleared strict CV and improved stress CV, but stress improvement is below the packaging gate."
elif selected_stacker != "base_only" and best_delta < -CONFIG["promotion_gate"]:
    recommendation = "do_not_package_stress_failed"
    recommendation_reason = "Residual stacking cleared strict CV but did not improve analog-distance stress CV."
elif selected_stacker != "base_only" and best_delta < 0:
    recommendation = "keep_as_minor_signal"
    recommendation_reason = "Residual stacking improved strict CV, but not enough to justify Kaggle packaging yet."
else:
    recommendation = "do_not_package"
    recommendation_reason = "Nested residual stacking did not improve the base inference-safe XGBoost path."

findings = pd.DataFrame([
    {"finding": "best_stacker", "value": selected_stacker},
    {"finding": "best_strict_rmse", "value": f"{float(best_row['rmse_mean']):.4f}"},
    {"finding": "base_only_strict_rmse", "value": f"{float(base_row['rmse_mean']):.4f}"},
    {"finding": "best_minus_base_rmse", "value": f"{best_delta:.4f}"},
    {"finding": "selected_stress_rmse", "value": f"{float(selected_stress_row['rmse_mean']):.4f}"},
    {"finding": "stress_base_only_rmse", "value": f"{float(stress_base_row['rmse_mean']):.4f}"},
    {"finding": "selected_stress_minus_base_rmse", "value": f"{stress_delta:.4f}"},
    {"finding": "best_minus_previous_legal_xgb_rmse", "value": f"{previous_delta:.4f}"},
    {"finding": "recommendation", "value": recommendation},
    {"finding": "recommendation_reason", "value": recommendation_reason},
])
findings_path = TABLE_DIR / "model__xgboost_residual_stacking_findings__2026-06-28.csv"
findings.to_csv(findings_path, index=False)
shutil.copy2(findings_path, RUN_DIR / findings_path.name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "xgboost_residual_stack",
    "model_name": "xgboost_residual_stacking",
    "cv_rmse_mean": float(best_row["rmse_mean"]),
    "cv_rmse_std": float(best_row["rmse_std"]),
    "train_time_s": "",
    "n_features": int(best_row["n_base_features"] + best_row["n_residual_features"]),
    "notes": recommendation_reason,
    "date": "2026-06-28",
    "validation_design": CONFIG["fold_column"],
    "local_rmse": float(best_row["rmse_mean"]),
    "stress_rmse": float(selected_stress_row["rmse_mean"]),
    "public_lb_rmse": "",
    "submission_ref": "",
}])
if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    keep_mask = ~comparison_df.get("model_name", pd.Series(dtype=str)).astype(str).eq("xgboost_residual_stacking")
    comparison_df = pd.concat([comparison_df.loc[keep_mask], comparison_row], ignore_index=True)
else:
    comparison_df = comparison_row
comparison_df.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_lines = [
    "# XGBoost Residual Stacking Report",
    "",
    f"Run id: `{RUN_ID}`",
    "",
    "## Result",
    "",
    f"- Best stacker: `{selected_stacker}`",
    f"- Best strict-CV RMSE: `{float(best_row['rmse_mean']):.4f}`",
    f"- Base-only strict-CV RMSE: `{float(base_row['rmse_mean']):.4f}`",
    f"- Strict best minus base RMSE: `{best_delta:.4f}`",
    f"- Selected stacker analog-distance stress RMSE: `{float(selected_stress_row['rmse_mean']):.4f}`",
    f"- Stress base-only RMSE: `{float(stress_base_row['rmse_mean']):.4f}`",
    f"- Stress selected minus base RMSE: `{stress_delta:.4f}`",
    f"- Best minus previous legal XGBoost RMSE: `{previous_delta:.4f}`",
    "",
    "## Decision",
    "",
    f"Recommendation: `{recommendation}`.",
    "",
    recommendation_reason,
    "",
    "## Interpretation",
    "",
    "The stackers are trained on inner out-of-fold residuals inside each held-out fold, so the result is a leakage-resistant test of whether residual shape remains after the base inference-safe XGBoost model. The analog-distance stress test checks whether the correction survives less similar wells.",
    "",
    "## Next Move",
    "",
]
if recommendation == "package_next":
    report_lines.append("Rerun once for reproducibility, then package the Ridge residual stacker for Kaggle.")
elif recommendation == "stress_positive_but_not_package_ready":
    report_lines.append("Tune the residual shrinkage and feature set before packaging, because stress improvement is positive but small.")
elif recommendation == "do_not_package_stress_failed":
    report_lines.append("Do not package this stacker; search for residual features that survive analog-distance stress.")
elif recommendation == "keep_as_minor_signal":
    report_lines.append("Keep the winning residual features as evidence, but search for a stronger structural residual signal before Kaggle.")
else:
    report_lines.append("Do not continue simple residual stacking; move to stronger stratigraphic or sequence-shape features.")

report_path = REPORT_DIR / "model__xgboost_residual_stacking__2026-06-28.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

findings

,finding,value
0,best_stacker,ridge_residual_s025
1,best_strict_rmse,13.2548
2,base_only_strict_rmse,13.4178
3,best_minus_base_rmse,-0.1630
4,selected_stress_rmse,13.5632
5,stress_base_only_rmse,13.7424
6,selected_stress_minus_base_rmse,-0.1792
7,best_minus_previous_legal_xgb_rmse,-0.1630
8,recommendation,package_after_reproducibility_check
9,recommendation_reason,Nested residual stacking cleared both strict-C...
